In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make src/ importable
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import preprocessing as pp

In [2]:
# Load the raw data and anomaly labels
NAB_ROOT = PROJECT_ROOT / "data" / "raw" / "NAB"
CSV_PATH = NAB_ROOT / "data" / "realKnownCause" / "machine_temperature_system_failure.csv"
LABELS_FILE = NAB_ROOT / "labels" / "combined_windows.json"
TARGET_FILE = "realKnownCause/machine_temperature_system_failure.csv"

df = pp.load_nab_stream(CSV_PATH)
anomaly_windows = pp.load_anomaly_windows(LABELS_FILE, TARGET_FILE)

print(f"Loaded {len(df):,} rows")
print(f"Anomaly windows: {len(anomaly_windows)}")
for i, (start, end) in enumerate(anomaly_windows, 1):
    print(f"  {i}. {start} -> {end}")

Loaded 22,695 rows
Anomaly windows: 4
  1. 2013-12-10 06:25:00 -> 2013-12-12 05:35:00
  2. 2013-12-15 17:50:00 -> 2013-12-17 17:00:00
  3. 2014-01-27 14:20:00 -> 2014-01-29 13:30:00
  4. 2014-02-07 14:55:00 -> 2014-02-09 14:05:00


In [3]:
# Split into train/val/test by time, then strip anomalies from training
train_df, val_df, test_df = pp.split_by_time(df, pp.DEFAULT_SPLIT)

# Only training needs anomaly removal — val and test keep their labels
train_df_clean = pp.remove_anomaly_windows(train_df, anomaly_windows)

print(f"Train (raw):   {len(train_df):,} rows")
print(f"Train (clean): {len(train_df_clean):,} rows  ({len(train_df) - len(train_df_clean):,} removed)")
print(f"Val:           {len(val_df):,} rows")
print(f"Test:          {len(test_df):,} rows")
print(f"\nTrain date range: {train_df_clean.timestamp.min()} to {train_df_clean.timestamp.max()}")
print(f"Val date range:   {val_df.timestamp.min()} to {val_df.timestamp.max()}")
print(f"Test date range:  {test_df.timestamp.min()} to {test_df.timestamp.max()}")

Train (raw):   15,309 rows
Train (clean): 14,175 rows  (1,134 removed)
Val:           576 rows
Test:          6,810 rows

Train date range: 2013-12-02 21:15:00 to 2014-01-24 23:55:00
Val date range:   2014-01-25 00:00:00 to 2014-01-26 23:55:00
Test date range:  2014-01-27 00:00:00 to 2014-02-19 15:25:00


In [4]:
# Fit z-score scaler on the CLEAN training data only
train_values = train_df_clean['value'].values
scaler = pp.fit_scaler(train_values)

print(f"Scaler fitted on training data:")
print(f"  mean = {scaler.mean:.4f}")
print(f"  std  = {scaler.std:.4f}")

# Apply the same scaler to all three splits
train_normalised = scaler.transform(train_df_clean['value'].values)
val_normalised   = scaler.transform(val_df['value'].values)
test_normalised  = scaler.transform(test_df['value'].values)

print(f"\nAfter normalisation:")
print(f"  Train: mean = {train_normalised.mean():.4f}, std = {train_normalised.std():.4f}")
print(f"  Val:   mean = {val_normalised.mean():.4f}, std = {val_normalised.std():.4f}")
print(f"  Test:  mean = {test_normalised.mean():.4f}, std = {test_normalised.std():.4f}")

Scaler fitted on training data:
  mean = 88.1135
  std  = 9.0416

After normalisation:
  Train: mean = -0.0000, std = 1.0000
  Val:   mean = -0.4258, std = 0.4194
  Test:  mean = -0.5461, std = 2.0701


In [5]:
# Window each split, respecting segment boundaries within training
X_train = pp.window_dataframe_by_segments(train_df_clean, scaler)
X_val   = pp.window_dataframe_by_segments(val_df,         scaler)
X_test  = pp.window_dataframe_by_segments(test_df,        scaler)

print(f"Window size: {pp.WINDOW_SIZE} readings ({pp.WINDOW_SIZE * 5} minutes)")
print(f"Stride:      {pp.WINDOW_STRIDE}")
print(f"\nWindowed shapes:")
print(f"  X_train: {X_train.shape}")
print(f"  X_val:   {X_val.shape}")
print(f"  X_test:  {X_test.shape}")

Window size: 60 readings (300 minutes)
Stride:      1

Windowed shapes:
  X_train: (13998, 60, 1)
  X_val:   (517, 60, 1)
  X_test:  (6751, 60, 1)
